# Importing Necessary modules

In [2]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from huggingface_hub import login
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path
import os
from dotenv import load_dotenv
from transformers.utils import logging
logging.set_verbosity_info()   # or logging.set_verbosity_debug()

/Users/kartikkaushal/Desktop/agentic-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/3g/9m7w77717kl6w9ftjmw665dh0000gq/T/ipykernel_23774/2175728275.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [2]:
load_dotenv()
hf_token = os.getenv("HF")
login(hf_token)

# Importing Model

In [3]:
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

In [4]:
#empty GPU (mps) memory
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    dtype = torch.float16
else:
    device = torch.device("cpu")
    dtype = torch.float32

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)

model.to(device)

print("Model device:", next(model.parameters()).device)

Using device: mps


[transformers] loading configuration file config.json from cache at /Users/kartikkaushal/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/snapshots/31b70e2e869a7173562077fd711b654946d38674/config.json
[transformers] Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 24,
  "num_key_value_heads": 32,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 130000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers.js_config": {
    "dtype": "q4",
    "kv_cache_dtype": {
      "fp16": "float16",
 

Model device: mps:0


In [19]:
phrase = input("Write a phrase: ")
inputs = tokenizer(phrase, return_tensors = "pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
inputs

{'input_ids': tensor([[6403,  665,  436,  253, 1838, 7706,  617,  436,  216,   34,   32,  929,
          1573]], device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [20]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

KeyboardInterrupt: 

# RAG Pipelining

## Create Pipeline

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

chat_model = ChatHuggingFace(llm=llm)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] loading configuration file config.json from cache at /Users/kartikkaushal/.cache/huggingface/hub/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/snapshots/31b70e2e869a7173562077fd711b654946d38674/config.json
[transformers] Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 24,
  "num_key_value_heads": 32,
  "pad_token_id"

## Search Engine

In [8]:
@tool
def search_local_database(query: str) -> str:
    """Use this tool to search the local vector database for context."""
    # (Your ChromaDB or FAISS retrieval code will go here later)
    return f"Retrieved documents about: {query}"

# Bind the tool to your Chat Model so Phi-4 knows it exists
model_with_tools = chat_model.bind_tools([search_local_database])

## Storing Data in ChromaDB

In [13]:
#Embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

[transformers] loading configuration file config.json from cache at /Users/kartikkaushal/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

[transformers]

In [ ]:
# -------------------------------
# Step 1: Load PDF
# -------------------------------

pdf_path = "test.pdf"   # Replace with your PDF file path

loader = PyPDFLoader(pdf_path)

# Each page becomes a Document object
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF.")

# -------------------------------
# Step 2: Split into chunks
# -------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} text chunks.")

# -------------------------------
# Step 4: Create ChromaDB
# -------------------------------

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db_storage"
)

# Save the database
vector_store.persist()

print(f"Successfully embedded and stored {len(chunks)} chunks in ChromaDB!")

Loaded 10 pages from the PDF.
Created 117 text chunks.


/var/folders/3g/9m7w77717kl6w9ftjmw665dh0000gq/T/ipykernel_71345/2017191586.py:31: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
[transformers] loading configuration file config.json from cache at /Users/kartikkaushal/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json
[transformers] Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
 

Successfully embedded and stored 117 chunks in ChromaDB!


/var/folders/3g/9m7w77717kl6w9ftjmw665dh0000gq/T/ipykernel_71345/2017191586.py:46: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [ ]:
pdf_folder = "pdfs" 
pdf_files = list(Path(pdf_folder).glob("*.pdf"))

all_documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    documents = loader.load()

    for doc in documents:
        doc.metadata["source_pdf"] = pdf_file.name
        doc.metadata["pdf_path"] = str(pdf_file)

    all_documents.extend(documents)

print(f"Loaded {len(all_documents)} pages from {len(pdf_files)} PDFs.")


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(all_documents)

print(f"Created {len(chunks)} text chunks.")


persist_dir = Path.cwd() / "chroma_db_storage"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(persist_dir)
)

vector_store.persist()

print(f"Successfully stored {len(chunks)} chunks from {len(pdf_files)} PDFs!")

Loaded 15 pages from 2 PDFs.
Created 126 text chunks.
Successfully stored 126 chunks from 2 PDFs!


/var/folders/3g/9m7w77717kl6w9ftjmw665dh0000gq/T/ipykernel_23774/3897257894.py:50: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [ ]:
persist_dir = Path.cwd() / "chroma_db_storage"

vector_store = Chroma(
    persist_directory=str(persist_dir),
    embedding_function=embedding_model
)

existing = vector_store.get(include=["metadatas"])

indexed_pdfs = set()

for metadata in existing["metadatas"]:
    if metadata and "source_pdf" in metadata:
        indexed_pdfs.add(metadata["source_pdf"])

print("Already indexed:")
print(indexed_pdfs)

pdf_folder = Path("pdfs")

new_pdfs = [
    pdf
    for pdf in pdf_folder.glob("*.pdf")
    if pdf.name not in indexed_pdfs
]

print(f"Found {len(new_pdfs)} new PDFs.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

new_chunks = []

for pdf_file in new_pdfs:

    print(f"Processing {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    for doc in docs:
        doc.metadata["source_pdf"] = pdf_file.name
        doc.metadata["pdf_path"] = str(pdf_file)

    chunks = text_splitter.split_documents(docs)
    new_chunks.extend(chunks)

print(f"Created {len(new_chunks)} new chunks.")

if new_chunks:
    vector_store.add_documents(new_chunks)
    vector_store.persist()
    print("Database updated successfully!")
else:
    print("No new PDFs to add.")

/var/folders/3g/9m7w77717kl6w9ftjmw665dh0000gq/T/ipykernel_23774/240783331.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


Already indexed:
{'2607.18065v1.pdf', '2607.16402v1-2.pdf'}
Found 4 new PDFs.
Processing 2607.17271v1.pdf
Processing 2607.18005v1.pdf
Processing 2607.16776v1.pdf
Processing 2607.17729v1.pdf
Created 617 new chunks.
Database updated successfully!


## Query Retrieval

In [21]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2}) # Get top 2 most relevant chunks

query = "How is machine learning being used ?"
results = retriever.invoke(query)

print(f"Query: {query}\n" + "="*40)
for i, doc in enumerate(results):
    print(f"Result {i+1} from {doc.metadata['source']}:")
    print(f"{doc.page_content}\n")

Query: How is machine learning being used ?
Result 1 from pdfs/2607.16776v1.pdf:
the model learns relevant physics rather than relying merely
on statistics.
4. The model outperforms the best single-feature physical
baseline by a substantial margin (PR-AUC 0.99 vs. 0.72),
demonstrating that binary formation requires nonlinear
combinations of features beyond simple thresholds.
5. With an inference speed of roughly 70,000 predictions per
second and a speedup of 400 times over direct N-body in-
tegration, the model is well-suited for rapid probability esti-

Result 2 from pdfs/2607.16776v1.pdf:
of the Pacific Conference Series, V ol. 534, Protostars and Planets VII, ed.
S. Inutsuka, Y . Aikawa, T. Muto, K. Tomida, & M. Tamura, 275
Pasquato, M., Trevisan, P., Askar, A., et al. 2024, ApJ, 965, 89
Pedregosa, F., Varoquaux, G., Gramfort, A., et al. 2011, Journal of Machine
Learning Research, 12, 2825
Poincaré, H. 1890, Acta Mathematica, 13, 1, oeuvres, tome VII, pp. 262-479
Portegies Zwart, S.